# IEEE-CID Fraud Detection — Data Preprocessing Pipeline
Full preprocessing pipeline for a deep learning (neural network) model.

**Order of steps:**
1. Imports
2. Load & Merge
3. Define column groups
4. EDA
5. Clean data (drop high-missing, impute, winsorise)
6. Train / Val / Test split (80 / 5 / 15) — **before feature engineering to prevent leakage**
7. Feature engineering on train only, then apply to val & test
8. Encode categorical features
9. Scale numerical features
10. Save outputs

## 1. Imports

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import pickle
import os

print('Libraries loaded successfully!')

Libraries loaded successfully!


## 2. Load & Merge
Left join on `TransactionID` — keeps all transactions even without an identity record.

In [17]:
transaction = pd.read_csv('data/train_transaction.csv')
identity    = pd.read_csv('data/train_identity.csv')

df = transaction.merge(identity, on='TransactionID', how='left')

print(f'Transaction rows : {len(transaction):,}')
print(f'Identity rows    : {len(identity):,}')
print(f'Merged rows      : {len(df):,}')
print(f'Merged columns   : {df.shape[1]}')

df.head()

Transaction rows : 590,540
Identity rows    : 144,233
Merged rows      : 590,540
Merged columns   : 434


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## 3. Define Column Groups

In [18]:
target    = 'isFraud'
drop_cols = ['TransactionID']

# Confirmed categorical columns per IEEE-CID dataset documentation
# Transaction: ProductCD, card1-card6, addr1, addr2, P_emaildomain, R_emaildomain, M1-M9
# Identity:    DeviceType, DeviceInfo, id_12-id_38
known_cat_cols = [
    'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo',
    'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19',
    'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27',
    'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35',
    'id_36', 'id_37', 'id_38'
]

cat_cols = []
num_cols = []

for col in df.columns:
    if col in ([target] + drop_cols):
        continue
    if df[col].isna().all():
        continue  # entirely null — will be dropped in cleaning
    if col in known_cat_cols:
        cat_cols.append(col)
    else:
        first_valid = df[col].dropna().iloc[0]
        try:
            float(first_valid)
            num_cols.append(col)
        except (ValueError, TypeError):
            cat_cols.append(col)  # unexpected string column — treat as categorical

print(f'Categorical columns : {len(cat_cols)}')
print(f'Numerical columns   : {len(num_cols)}')
print(f'Target              : {target}')
print(f'\nCategorical: {cat_cols}')

Categorical columns : 49
Numerical columns   : 383
Target              : isFraud

Categorical: ['ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


## 4. EDA

In [19]:
print('All columns in the dataset:')
print(df.columns.tolist())

All columns in the dataset:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V

In [20]:
fraud_percentage = df['isFraud'].mean() * 100
print(f'Percentage of fraudulent transactions: {fraud_percentage:.4f}%')

fraud_counts = df['isFraud'].value_counts()
print('Number of transactions by class:')
print(fraud_counts)

Percentage of fraudulent transactions: 3.4990%
Number of transactions by class:
isFraud
0    569877
1     20663
Name: count, dtype: int64


In [21]:
# Standardise missing value representations
missing_values = ['', 'NA', 'NaN', 'None']
df.replace(missing_values, np.nan, inplace=True)

# Missing value summary
nan_summary = pd.DataFrame({
    'nan_count':      df.isnull().sum(),
    'nan_percentage': df.isnull().mean() * 100
})
nan_summary = nan_summary[nan_summary['nan_count'] > 0].sort_values('nan_percentage', ascending=False)
print(nan_summary)

       nan_count  nan_percentage
id_24     585793       99.196159
id_25     585408       99.130965
id_07     585385       99.127070
id_08     585385       99.127070
id_21     585381       99.126393
...          ...             ...
V285          12        0.002032
V284          12        0.002032
V280          12        0.002032
V279          12        0.002032
V312          12        0.002032

[414 rows x 2 columns]


In [22]:
# Inspect high-missing numeric columns and their correlations
tbd_columns = nan_summary[nan_summary['nan_percentage'] > 75].index.tolist()
print(f'Columns with >75% missing: {len(tbd_columns)}')

numeric_tbd_columns = df[tbd_columns].select_dtypes(include='number').columns.tolist()
categorical_tbd_columns = df[tbd_columns].select_dtypes(include=['object', 'category']).columns.tolist()

print(f'  Numeric   : {len(numeric_tbd_columns)}')
print(f'  Categorical: {len(categorical_tbd_columns)}')

correlation_matrix = df[numeric_tbd_columns].corr()
print('\nCorrelation matrix for high-missing numeric columns:')
print(correlation_matrix)

Columns with >75% missing: 208
  Numeric   : 190
  Categorical: 18

Correlation matrix for high-missing numeric columns:
          id_24     id_25     id_07     id_08     id_21     id_26     id_22  \
id_24  1.000000 -0.030902 -0.070752 -0.011849  0.220933  0.086002  0.014067   
id_25 -0.030902  1.000000  0.037649 -0.003628 -0.147694  0.011508  0.182643   
id_07 -0.070752  0.037649  1.000000 -0.094086 -0.188250 -0.131638 -0.277448   
id_08 -0.011849 -0.003628 -0.094086  1.000000  0.085533  0.037212  0.143301   
id_21  0.220933 -0.147694 -0.188250  0.085533  1.000000  0.053721  0.070591   
...         ...       ...       ...       ...       ...       ...       ...   
V259   0.000209 -0.000867  0.025090  0.065377 -0.060745  0.099018 -0.021577   
V271  -0.009570 -0.019333 -0.043122  0.052596 -0.040013  0.020131 -0.013223   
V272  -0.011819 -0.020324 -0.039184  0.060007 -0.039551  0.020837 -0.012861   
V238  -0.036288  0.009524  0.005660  0.009542 -0.040539 -0.072507 -0.027627   
id_01 -0.1

## 5. Clean Data
- Drop columns with **>75% missing** (too sparse to be useful)
- **Median impute** remaining numerical NaNs — neural networks cannot handle NaN
- Fill categorical NaNs with `'missing'`
- **Winsorise** numerical outliers at 1st/99th percentile — neural networks are sensitive to extreme values
- Drop `TransactionID`

> Cleaning is done before the split so the missingness threshold is computed on the full dataset, which is fine since it is a structural property of the data, not a value learned from it.

In [23]:
# ── Drop high-missingness columns ─────────────────────────────────────────────
missing_rate = df[num_cols + cat_cols].isnull().mean()
high_missing = missing_rate[missing_rate > 0.90].index.tolist()

print(f'Dropping {len(high_missing)} columns with > 90% missing values')
df.drop(columns=high_missing, inplace=True)
cat_cols = [c for c in cat_cols if c not in high_missing]
num_cols = [c for c in num_cols if c not in high_missing]

# ── Impute numerical NaNs with median ────────────────────────────────────────
for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# ── Impute categorical NaNs with 'missing' ───────────────────────────────────
for col in cat_cols:
    df[col] = df[col].astype(str).replace('nan', 'missing')

# ── Winsorise numerical outliers at 1st / 99th percentile ───────────────────
for col in num_cols:
    low  = df[col].quantile(0.01)
    high = df[col].quantile(0.99)
    df[col] = df[col].clip(low, high)

# ── Drop ID column ───────────────────────────────────────────────────────────
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

print(f'Shape after cleaning : {df.shape}')
print(f'Remaining nulls      : {df.isnull().sum().sum()}')

Dropping 12 columns with > 90% missing values
Shape after cleaning : (590540, 421)
Remaining nulls      : 0


## 6. Train / Val / Test Split (80 / 5 / 15)
Split **before** feature engineering so that aggregation and frequency features are computed only from training data — no leakage into val or test.
Stratification ensures all three splits keep the same ~3.5% fraud rate.

In [24]:
# Step 1: split off 15% test
train_val, test = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df[target]
)

# Step 2: split remaining 85% → 80% train, 5% val
# 5% of total = 5/85 ≈ 0.0588 of train_val
train, val = train_test_split(
    train_val, test_size=0.0588, random_state=42, stratify=train_val[target]
)

# Reset indices for clean indexing
train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

print(f'Train      : {len(train):,} rows ({len(train)/len(df):.1%}) | fraud rate: {train[target].mean():.3%}')
print(f'Validation : {len(val):,} rows ({len(val)/len(df):.1%})  | fraud rate: {val[target].mean():.3%}')
print(f'Test       : {len(test):,} rows ({len(test)/len(df):.1%}) | fraud rate: {test[target].mean():.3%}')

Train      : 472,443 rows (80.0%) | fraud rate: 3.499%
Validation : 29,516 rows (5.0%)  | fraud rate: 3.500%
Test       : 88,581 rows (15.0%) | fraud rate: 3.498%


## 7. Feature Engineering

### Why val and test still receive new columns
The model is trained expecting a fixed set of input columns (e.g. `card1_freq`, `card1_amt_mean`, etc.).
If val and test don't have those same columns, the model cannot make predictions on them — it would be like giving someone a form with fields missing.

So the rule is:
- **COMPUTE** every statistic and mapping **from train only** — val/test never contribute to any calculation
- **STAMP** those train-derived values onto val and test as a lookup — no new information is learned from val/test

This is identical to how `StandardScaler` works: you `fit` on train (compute mean & std from train only), then `transform` val and test (stamp those train-derived values onto them).

### Features added
- **Time-based**: hour of day, day-of-week proxy, weekend flag — computed from the timestamp directly, no train-fitting needed
- **Frequency**: how often each card / email domain appears — *fit on train*, looked up for val/test
- **Aggregation**: per-card transaction amount mean/std/count, deviation from card mean — *fit on train*, looked up for val/test
- **Transaction amount**: log transform, cents (decimal part) — pure math, no train-fitting needed
- **Interaction**: card+address and card+email combination frequency — *fit on train*, looked up for val/test
- **M-column summary**: count of T / F / null across M1–M9 — pure count, no train-fitting needed

In [25]:
def apply_feature_engineering(train_df, val_df, test_df):
    """
    All statistics/mappings are COMPUTED from train_df only.
    Val and test only receive the resulting column — they never contribute
    to any calculation. Unseen values in val/test are filled with 0.
    """

    # ── Time-based features ──────────────────────────────────────────────────
    # These are pure math on the timestamp — no fitting needed,
    # so it is safe to apply directly to all three splits.
    for split in [train_df, val_df, test_df]:
        split['Transaction_hour']    = (split['TransactionDT'] // 3600) % 24
        split['Transaction_day']     = (split['TransactionDT'] // 86400) % 7
        split['Transaction_weekend'] = (split['Transaction_day'] >= 5).astype(int)

    # ── Card frequency ───────────────────────────────────────────────────────
    # STEP 1 — FIT: count how often each card appears in TRAIN only
    # STEP 2 — APPLY: look up each card's train-frequency in val/test
    #          Cards not seen in train get 0 (they have no known frequency)
    card_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6']
    for col in card_cols:
        if col not in train_df.columns:
            continue
        freq_map = train_df[col].value_counts().to_dict()  # FIT on train
        for split in [train_df, val_df, test_df]:          # APPLY to all
            split[f'{col}_freq'] = split[col].map(freq_map).fillna(0)

    # ── Email domain frequency ───────────────────────────────────────────────
    # Same pattern: FIT on train, APPLY to all three
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col not in train_df.columns:
            continue
        freq_map = train_df[col].value_counts().to_dict()  # FIT on train
        for split in [train_df, val_df, test_df]:          # APPLY to all
            split[f'{col}_freq'] = split[col].map(freq_map).fillna(0)

    # ── Per-card transaction amount stats ────────────────────────────────────
    # STEP 1 — FIT: compute mean/std/count of TransactionAmt per card in TRAIN only
    # STEP 2 — APPLY: merge those train-derived stats onto val/test as a lookup table
    #          Cards not seen in train get 0 for all stat columns
    card_num_cols = ['card1', 'card2', 'card3', 'card5']
    for col in card_num_cols:
        if col not in train_df.columns:
            continue

        # FIT — computed from train only
        stats = train_df.groupby(col)['TransactionAmt'].agg(['mean', 'std', 'count'])
        stats.columns = [f'{col}_amt_mean', f'{col}_amt_std', f'{col}_count']

        # APPLY — merge train-derived stats onto each split as a lookup
        for split_name, split_ref in [('train', train_df), ('val', val_df), ('test', test_df)]:
            merged = split_ref.merge(stats, on=col, how='left')
            merged[f'{col}_amt_diff_from_mean'] = merged['TransactionAmt'] - merged[f'{col}_amt_mean']
            merged[f'{col}_amt_ratio_to_mean']  = merged['TransactionAmt'] / (merged[f'{col}_amt_mean'] + 1e-9)
            # Fill 0 for cards in val/test not seen in train
            for feat in [f'{col}_amt_mean', f'{col}_amt_std', f'{col}_count',
                         f'{col}_amt_diff_from_mean', f'{col}_amt_ratio_to_mean']:
                merged[feat] = merged[feat].fillna(0)
            if split_name == 'train': train_df = merged
            elif split_name == 'val': val_df   = merged
            else:                     test_df  = merged

    # ── Transaction amount features ──────────────────────────────────────────
    # Pure math — no fitting needed, safe to apply to all three splits directly
    for split in [train_df, val_df, test_df]:
        split['TransactionAmt_log']   = np.log1p(split['TransactionAmt'])
        split['TransactionAmt_cents'] = (split['TransactionAmt'] % 1).round(2)

    # ── Interaction features: card + address / email ─────────────────────────
    # STEP 1 — FIT: count how often each (card, address) pair appears in TRAIN only
    # STEP 2 — APPLY: look up each pair's train-frequency in val/test
    #          Pairs not seen in train get 0
    for col in card_cols:
        if col not in train_df.columns:
            continue
        if 'addr1' in train_df.columns:
            combo_train = train_df[col].astype(str) + '_' + train_df['addr1'].astype(str)
            freq_map = combo_train.value_counts().to_dict()  # FIT on train
            for split in [train_df, val_df, test_df]:        # APPLY to all
                combo = split[col].astype(str) + '_' + split['addr1'].astype(str)
                split[f'{col}_addr1_freq'] = combo.map(freq_map).fillna(0)

        if 'P_emaildomain' in train_df.columns:
            combo_train = train_df[col].astype(str) + '_' + train_df['P_emaildomain'].astype(str)
            freq_map = combo_train.value_counts().to_dict()  # FIT on train
            for split in [train_df, val_df, test_df]:        # APPLY to all
                combo = split[col].astype(str) + '_' + split['P_emaildomain'].astype(str)
                split[f'{col}_email_freq'] = combo.map(freq_map).fillna(0)

    # ── M-column summary features ─────────────────────────────────────────────
    # Pure counts of T/F/null — no fitting needed, safe to apply to all splits
    m_cols = [c for c in train_df.columns if c.startswith('M') and c[1:].isdigit()]
    if m_cols:
        for split in [train_df, val_df, test_df]:
            split['M_true_count']  = (split[m_cols] == 'T').sum(axis=1)
            split['M_false_count'] = (split[m_cols] == 'F').sum(axis=1)
            split['M_null_count']  = split[m_cols].isnull().sum(axis=1)

    return train_df, val_df, test_df


train, val, test = apply_feature_engineering(train, val, test)

print(f'Train shape after feature engineering : {train.shape}')
print(f'Val shape after feature engineering   : {val.shape}')
print(f'Test shape after feature engineering  : {test.shape}')

Train shape after feature engineering : (472443, 469)
Val shape after feature engineering   : (29516, 469)
Test shape after feature engineering  : (88581, 469)


In [26]:
# Refresh both lists against what actually exists in train after feature engineering.
# cat_cols may still contain columns that were dropped in the cleaning step,
# so we filter it first — otherwise those ghost columns would accidentally
# exclude real numerical columns from num_cols.
cat_cols = [c for c in cat_cols if c in train.columns]
num_cols = [c for c in train.columns if c not in set(cat_cols + [target])]

# Impute any NaNs introduced by merges in val/test (unseen card values)
for col in num_cols:
    for split in [train, val, test]:
        if split[col].isnull().any():
            split[col] = split[col].fillna(split[col].median())

print(f'Updated — Categorical: {len(cat_cols)}, Numerical: {len(num_cols)}')

Updated — Categorical: 41, Numerical: 427


## 8. Encode Categorical Features
- `card4` and `card6` (e.g. visa/mastercard) are **one-hot encoded** — low cardinality, no meaningful order
- All other categoricals are **label encoded** to integers for use with `nn.Embedding` layers
- Encoders are **fit on train only**, then applied to val and test
- Unseen categories in val/test are mapped to an out-of-vocabulary (OOV) index

In [27]:
encoders = {}

for col in cat_cols:
    if col not in train.columns:
        continue

    le = LabelEncoder()
    le.fit(train[col])

    class_map = {c: i for i, c in enumerate(le.classes_)}
    oov_index = len(le.classes_)

    for split in [train, val, test]:
        split[col] = split[col].map(class_map).fillna(oov_index).astype(int)

    encoders[col] = le

vocab_sizes = {col: len(le.classes_) + 1 for col, le in encoders.items()}

print(f'Label encoded {len(encoders)} categorical columns.')
print('\nVocabulary sizes (for Embedding layers):')
for col, size in vocab_sizes.items():
    print(f'  {col}: {size}')

Label encoded 41 categorical columns.

Vocabulary sizes (for Embedding layers):
  ProductCD: 6
  card1: 12819
  card2: 502
  card3: 115
  card4: 6
  card5: 116
  card6: 6
  addr1: 307
  addr2: 71
  P_emaildomain: 61
  R_emaildomain: 62
  M1: 4
  M2: 4
  M3: 4
  M4: 5
  M5: 4
  M6: 4
  M7: 4
  M8: 4
  M9: 4
  id_12: 4
  id_13: 55
  id_14: 26
  id_15: 5
  id_16: 4
  id_17: 103
  id_19: 517
  id_20: 386
  id_28: 4
  id_29: 4
  id_30: 77
  id_31: 128
  id_32: 6
  id_33: 233
  id_34: 6
  id_35: 4
  id_36: 4
  id_37: 4
  id_38: 4
  DeviceType: 4
  DeviceInfo: 1680


## 9. Scale Numerical Features
`StandardScaler` fit on **train only**, applied to val and test. Essential for neural network convergence.

In [28]:
non_numeric = set([target] + list(encoders.keys()))
num_cols_to_scale = [c for c in train.columns if c not in non_numeric]

scaler = StandardScaler()
train[num_cols_to_scale] = scaler.fit_transform(train[num_cols_to_scale])
val[num_cols_to_scale]   = scaler.transform(val[num_cols_to_scale])
test[num_cols_to_scale]  = scaler.transform(test[num_cols_to_scale])

print(f'Scaled {len(num_cols_to_scale)} numerical columns.')


Scaled 427 numerical columns.


## 10. Final Check

In [29]:
print('=== Final Dataset Summary ===')
print(f'Train      : {train.shape} | nulls: {train.isnull().sum().sum()}')
print(f'Validation : {val.shape}   | nulls: {val.isnull().sum().sum()}')
print(f'Test       : {test.shape}  | nulls: {test.isnull().sum().sum()}')
print(f'\nFraud rate — Train: {train[target].mean():.3%} | Val: {val[target].mean():.3%} | Test: {test[target].mean():.3%}')
print(f'\nNumerical features  : {len(num_cols_to_scale)}')
print(f'Label-enc features  : {len(encoders)}')
#print(f'One-hot features    : {len(encoded_cat_cols)}')

train.head()

=== Final Dataset Summary ===
Train      : (472443, 469) | nulls: 0
Validation : (29516, 469)   | nulls: 0
Test       : (88581, 469)  | nulls: 0

Fraud rate — Train: 3.499% | Val: 3.500% | Test: 3.498%

Numerical features  : 427
Label-enc features  : 41


,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,card3_email_freq,card4_addr1_freq,card4_email_freq,card5_addr1_freq,card5_email_freq,card6_addr1_freq,card6_email_freq,M_true_count,M_false_count,M_null_count
0,0,-1.284509,-0.604289,1,2413,184,41,4,103,2,...,-0.968842,-0.293013,-0.853909,0.078955,-0.518691,-0.392513,-0.953699,-1.078728,-0.932218,0.0
1,0,-1.324847,-0.456764,1,1853,69,41,4,103,1,...,-1.100228,-0.307886,-0.746951,0.090709,-0.523547,-0.965416,-1.015932,-1.078728,-0.932218,0.0
2,0,0.270974,-0.114800,4,10248,105,41,2,21,2,...,1.258637,-0.946576,0.100143,-0.934464,-0.797322,-0.809333,1.405825,1.496585,-0.296365,0.0
3,0,-0.191930,-0.061396,4,1779,219,41,2,76,2,...,-0.072422,-1.012351,-0.691330,-0.933189,-0.874764,-0.893255,-0.045421,1.496585,0.339488,0.0
4,0,-0.417743,0.463793,4,7539,0,41,4,103,2,...,1.258637,1.644214,1.545978,2.346968,1.858708,1.266592,1.405825,0.208928,2.247047,0.0


## 11. Save Outputs

In [ ]:
os.makedirs('preprocessed', exist_ok=True)

train.to_csv('preprocessed/train.csv', index=False)
val.to_csv('preprocessed/val.csv',     index=False)
test.to_csv('preprocessed/test.csv',   index=False)

with open('preprocessed/scaler.pkl',   'wb') as f: pickle.dump(scaler,   f)
with open('preprocessed/encoders.pkl', 'wb') as f: pickle.dump(encoders, f)

metadata = {
    'cat_cols'         : list(encoders.keys()),
    'num_cols'         : num_cols_to_scale,
    'vocab_sizes'      : vocab_sizes,
    'target'           : target
}
with open('preprocessed/column_metadata.pkl', 'wb') as f: pickle.dump(metadata, f)

print('Saved to ./preprocessed/')
print('  train.csv, val.csv, test.csv')
print('  scaler.pkl, encoders.pkl, column_metadata.pkl')

Saved to ./preprocessed/
  train.csv, val.csv, test.csv
  scaler.pkl, encoders.pkl, column_metadata.pkl


: 